In [ ]:
# ============================================================
# 1일차 오전: BPE → Self-Attention 시각화 → Positional Encoding
# 프레임워크: Keras (TF 2.x)  |  토크나이저: HuggingFace tokenizers
# 목표: 오후 Decoder-only 추론의 개념 기반 구축
# ============================================================

!pip install -q tokenizers transformers tensorflow matplotlib seaborn koreanize-matplotlib

In [2]:
# ────────────────────────────────────────────────────────────
# 블록 3: Positional Encoding (30분)
# sin/cos 패턴 시각화 + 위치 정보 역할 이해
# ────────────────────────────────────────────────────────────
import numpy as np
import tensorflow as tf
from tensorflow import keras
import koreanize_matplotlib  # matplotlib에서 한글 깨짐 방지
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

## 3-1. Positional Encoding 구현
def positional_encoding(max_len, d_model):
    """
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    positions = np.arange(max_len)[:, np.newaxis]      # (max_len, 1)
    dims      = np.arange(d_model)[np.newaxis, :]      # (1, d_model)

    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    pe = np.where(dims % 2 == 0, np.sin(angles), np.cos(angles))
    return pe   # (max_len, d_model)

## 3-2. 시각화 1: PE 패턴 히트맵
max_len, d_model_pe = 50, 64
pe = positional_encoding(max_len, d_model_pe)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(pe, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_xlabel("임베딩 차원 (d_model)")
axes[0].set_ylabel("토큰 위치 (position)")
axes[0].set_title("Positional Encoding 패턴\n(빨강=+1, 파랑=-1)", fontsize=12)
plt.colorbar(im, ax=axes[0])

## 3-3. 시각화 2: 특정 차원의 sin/cos 파형
dims_to_plot = [0, 4, 16, 32]
colors = ['#E24B4A', '#1D9E75', '#378ADD', '#EF9F27']
for d, c in zip(dims_to_plot, colors):
    axes[1].plot(pe[:, d], label=f"dim {d}", color=c, linewidth=1.5)

axes[1].set_xlabel("토큰 위치 (position)")
axes[1].set_ylabel("PE 값")
axes[1].set_title("차원별 sin/cos 파형\n(차원마다 다른 주파수)", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')

plt.suptitle("Positional Encoding — 위치 정보를 벡터에 추가하는 방법", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("positional_encoding.png", dpi=150, bbox_inches='tight')
plt.show()

## 3-4. 위치 유사도 — 가까운 위치끼리 유사한 PE 벡터
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

pos_pairs = [(0,1), (0,5), (0,25), (0,49)]
print("\n위치 간 PE 코사인 유사도:")
print("(가까운 위치일수록 유사한 벡터 → 모델이 상대적 위치 학습 가능)")
print("-" * 40)
for p1, p2 in pos_pairs:
    sim = cosine_sim(pe[p1], pe[p2])
    bar = "-" * int(sim * 20)
    print(f"  위치 {p1:2d} ↔ 위치 {p2:2d}: {sim:.3f}  {bar}")

## 3-5. 임베딩 + PE 결합 시각화
print("\n★ 핵심 요약")
print("  Self-Attention 자체는 순서 정보가 없음 (집합 연산)")
print("  → PE를 임베딩에 더해 위치 정보 주입")
print("  → 토큰 벡터 = 의미(임베딩) + 위치(PE)")
print("  → 오후 Decoder: 이 벡터로 '다음 위치의 토큰'을 예측")

ModuleNotFoundError: No module named 'tensorflow'